# Bronze — ingestão com Auto Loader

Lê os lotes que o simulador grava em `olist.landing.raw_data/<entidade>/sim_date=AAAA-MM-DD/` e acrescenta os registros novos em `olist.bronze.<entidade>` (Delta, append-only), uma tabela por entidade.

**Parâmetros**

| Widget | Padrão | Uso |
|---|---|---|
| `run_id` | vazio | identificador da execução, gravado em `_run_id`; no job recebe `{{job.run_id}}`, vazio gera um UUID |

**Regras**

- Cada entidade tem um stream próprio, com checkpoint e schema em `/Volumes/olist/ops/checkpoints/bronze/<entidade>/`. O checkpoint garante que cada arquivo seja lido uma única vez: **apagar o checkpoint duplica a bronze**.
- `trigger(availableNow=True)`: processa os arquivos novos e encerra.
- Todas as colunas de negócio como `string`; a tipagem fica na silver.

Desenho: `docs/design/fase-1-simulador-bronze.md` (decisões B1 a B6).

In [0]:
# O QUE FAZ: cria o parâmetro run_id, exibido como caixa de texto no topo do notebook.
#   A bronze não tem parâmetro de dias: o checkpoint de cada stream já sabe quais arquivos
#   foram lidos, então cada execução lê só o que é novo.
#
# FUNÇÕES:
# - dbutils.widgets.text(nome, valor_padrão, rótulo): cria um widget de texto (se já existe,
#   nada muda). O valor é lido com dbutils.widgets.get(nome) e sempre chega como string.
dbutils.widgets.text("run_id", "", "ID da execução")

In [0]:
# O QUE FAZ: define as constantes da bronze: pasta de origem (landing), pasta dos checkpoints,
#   schema de destino e a lista das 10 entidades. Nenhum dado é lido aqui.
#
# RECURSOS:
# - f"...{CATALOG}...": f-string; todos os caminhos derivam do catálogo.
# - /Volumes/<catálogo>/<schema>/<volume>: caminho de arquivos de um Volume do Unity Catalog.
# - tuple (...): como uma lista, mas não pode ser alterada depois de criada, o que serve bem para
#   uma configuração fixa. Incluir uma entidade nova = acrescentar uma linha (desenho B1).
CATALOG = "olist"

RAW_DIR = f"/Volumes/{CATALOG}/landing/raw_data"
CHECKPOINT_DIR = f"/Volumes/{CATALOG}/ops/checkpoints/bronze"
BRONZE_SCHEMA = f"{CATALOG}.bronze"

# As mesmas 10 entidades gravadas pelo simulador
ENTITIES = (
    "orders",
    "order_status_events",
    "order_items",
    "order_payments",
    "order_reviews",
    "customers",
    "sellers",
    "products",
    "geolocation",
    "category_translation",
)

In [0]:
# O QUE FAZ: declara as funções que derivam, do nome da entidade, todos os caminhos do stream,
#   define o run_id e imprime o mapeamento entidade -> origem -> tabela para conferência.
#
# FUNÇÕES DECLARADAS (puras: só montam texto, podem ser testadas sem Spark):
# - source_path(entity): pasta lida pelo Auto Loader: raw_data/<entidade>.
# - checkpoint_path(entity): checkpoint exclusivo do stream da entidade. Guarda quais arquivos
#   já foram processados e o progresso de cada micro-batch.
# - schema_path(entity): onde o Auto Loader guarda o schema inferido e suas versões; fica dentro
#   da pasta do checkpoint (desenho B3).
# - table_name(entity): nome completo da tabela de destino: olist.bronze.<entidade>.
#
# FUNÇÕES NATIVAS:
# - dbutils.widgets.get(nome): lê o valor do widget (string).
# - str.strip(): remove espaços do início e do fim; "texto or outro": se vazio, usa o outro.
# - uuid.uuid4(): identificador aleatório único, usado quando o run_id não é informado.
# - f"{entity:22}": formatação; ocupa 22 caracteres (alinha as colunas do print).
# - "\n" no texto: quebra de linha.
import uuid


def source_path(entity: str) -> str:
    """Pasta lida pelo Auto Loader (contém as subpastas sim_date=...)."""
    return f"{RAW_DIR}/{entity}"


def checkpoint_path(entity: str) -> str:
    """Estado do stream: arquivos já processados e progresso dos micro-batches."""
    return f"{CHECKPOINT_DIR}/{entity}"


def schema_path(entity: str) -> str:
    """Schema inferido e suas versões, dentro da pasta do checkpoint (B3)."""
    return f"{checkpoint_path(entity)}/_schema"


def table_name(entity: str) -> str:
    return f"{BRONZE_SCHEMA}.{entity}"


RUN_ID = dbutils.widgets.get("run_id").strip() or str(uuid.uuid4())

print(f"run_id: {RUN_ID}\n")
for entity in ENTITIES:
    print(f"{entity:22} {source_path(entity):55} → {table_name(entity)}")

## Leitura com Auto Loader e colunas de auditoria

Cada entidade é lida como um stream do Auto Loader (`cloudFiles`) sobre a sua pasta em `raw_data`:

- **Schema inferido, tudo `string`** (`inferColumnTypes = false`), guardado em `<checkpoint>/_schema/`.
- **Coluna nova na origem:** `addNewColumns` acrescenta a coluna ao schema e interrompe o stream; a nova tentativa fica na Parte 3.
- **Valor fora do schema:** vai para `_rescued_data`.
- **Linha que não é JSON válido:** guardada inteira em `_corrupt_record`, com as demais colunas nulas (ADR 0004). Sem essa coluna, a linha viraria um registro todo nulo, sem o texto original.
- **Arquivo regravado com o mesmo nome:** não é lido de novo (`allowOverwrites = false`).
- **`sim_date`:** o simulador grava com `partitionBy`, então o valor existe só no nome da pasta (`sim_date=AAAA-MM-DD`) e é lido de lá.

Colunas de auditoria (desenho B5): `_ingested_at`, `_source_file`, `_source_file_modified_at`, `_run_id`.

In [0]:
# O QUE FAZ: reúne num dicionário as opções do Auto Loader comuns a todas as entidades.
#   Algumas são o valor padrão e foram declaradas mesmo assim: deixam a decisão do desenho
#   explícita no código e protegem contra uma mudança de padrão em versões futuras.
#
# OPÇÕES:
# - cloudFiles.format = "json": formato dos arquivos de origem (JSON Lines, um registro por linha).
# - cloudFiles.partitionColumns = "sim_date": lê a coluna sim_date do nome da pasta
#   (sim_date=AAAA-MM-DD), no estilo Hive. Partição também chega como string.
# - cloudFiles.schemaEvolutionMode = "addNewColumns": coluna nova entra no schema; o stream para
#   com UnknownFieldException e, ao reiniciar, segue com o schema novo (desenho B4).
# - cloudFiles.inferColumnTypes = "false": não tenta descobrir tipos; todas as colunas são string.
# - cloudFiles.allowOverwrites = "false": arquivo já processado não é relido, mesmo se regravado.
# - cloudFiles.schemaHints = "_corrupt_record string": acrescenta a coluna _corrupt_record ao
#   schema inferido, em todas as entidades (ADR 0004).
# - columnNameOfCorruptRecord = "_corrupt_record": coluna que recebe o texto da linha que não pôde
#   ser lida como JSON. É opção do leitor JSON, por isso não tem o prefixo cloudFiles. O
#   _rescued_data não cobre esse caso: ele só guarda colunas desconhecidas, tipos divergentes e
#   diferenças de maiúsculas.
#
# Os valores são strings ("false"), porque as opções de leitura do Spark são sempre texto.
AUTOLOADER_OPTIONS = {
    "cloudFiles.format": "json",
    "cloudFiles.partitionColumns": "sim_date",
    "cloudFiles.schemaEvolutionMode": "addNewColumns",
    "cloudFiles.inferColumnTypes": "false",
    "cloudFiles.allowOverwrites": "false",
    "cloudFiles.schemaHints": "_corrupt_record string",
    "columnNameOfCorruptRecord": "_corrupt_record",
}

In [0]:
# O QUE FAZ: declara a leitura em stream de uma entidade e a função que acrescenta as colunas
#   de auditoria. Nenhuma das duas processa dados: elas só montam o plano do stream.
#
# FUNÇÕES DECLARADAS:
# - read_entity_stream(entity): devolve um DataFrame de streaming do Auto Loader sobre
#   raw_data/<entidade>, com o schema guardado em <checkpoint>/_schema.
# - add_audit_columns(df, run_id): função pura (DataFrame -> DataFrame) que acrescenta as 4
#   colunas de auditoria. Funciona igual em DataFrame de streaming ou batch lido de arquivos.
#
# FUNÇÕES NATIVAS:
# - spark.readStream: leitor de streaming (o equivalente incremental de spark.read).
# - .format("cloudFiles"): usa o Auto Loader como fonte.
# - .options(**dicionario): aplica todas as opções do dicionário de uma vez; "**" desempacota
#   o dicionário em pares chave=valor.
# - .option("cloudFiles.schemaLocation", caminho): pasta onde o Auto Loader grava o schema
#   inferido e as versões seguintes. É obrigatória quando o schema é inferido.
# - .load(caminho): pasta monitorada. Na definição do stream, o Auto Loader infere o schema
#   (amostra dos arquivos) ou reaproveita o que já está em schemaLocation.
# - F.current_timestamp(): data/hora do processamento; em streaming, é avaliada a cada
#   micro-batch.
# - F.col("_metadata.file_path") / ("_metadata.file_modification_time"): campos da coluna
#   oculta _metadata, disponível em toda leitura de arquivos; precisa ser selecionada de forma
#   explícita. Substitui a função obsoleta input_file_name().
# - F.lit(run_id): o mesmo run_id em todas as linhas desta execução.
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def read_entity_stream(entity: str) -> DataFrame:
    """Stream do Auto Loader sobre a pasta da entidade; todas as colunas como string."""
    return (
        spark.readStream
        .format("cloudFiles")
        .options(**AUTOLOADER_OPTIONS)
        .option("cloudFiles.schemaLocation", schema_path(entity))
        .load(source_path(entity))
    )


def add_audit_columns(df: DataFrame, run_id: str) -> DataFrame:
    """Colunas de rastreabilidade (desenho B5): quando chegou, de qual arquivo e qual execução gravou."""
    return df.withColumns({
        "_ingested_at": F.current_timestamp(),
        "_source_file": F.col("_metadata.file_path"),
        "_source_file_modified_at": F.col("_metadata.file_modification_time"),
        "_run_id": F.lit(run_id),
    })

In [0]:
# O QUE FAZ: monta o stream de orders com as colunas de auditoria e mostra o schema resultante,
#   sem gravar nenhum dado na bronze.
#   Efeito colateral esperado: o Auto Loader infere o schema e grava a primeira versão em
#   /Volumes/olist/ops/checkpoints/bronze/orders/_schema/. É o mesmo local que o stream da
#   Parte 3 vai usar, então não atrapalha: ele só reaproveita o schema já inferido.
#
# FUNÇÕES NATIVAS:
# - df.isStreaming: True se o DataFrame é de streaming (só roda com writeStream, não com count).
# - df.printSchema(): mostra as colunas e os tipos em árvore. Não executa o stream.
preview = add_audit_columns(read_entity_stream("orders"), RUN_ID)

print(f"isStreaming: {preview.isStreaming}")
preview.printSchema()

## Ingestão por entidade

`ingest_entity` grava os arquivos novos de uma entidade em `olist.bronze.<entidade>`:

- **Delta, append-only:** cada registro lido vira uma linha nova; nada é atualizado ou apagado.
- **`trigger(availableNow=True)`:** processa tudo o que chegou desde a última execução e encerra (decisão B2). Sem trigger declarado, o serverless recusa o stream.
- **Tabela criada na primeira execução** pelo próprio stream (`toTable`).
- **Mudança de schema:** quando o Auto Loader encontra uma coluna nova, o stream para com `UnknownFieldException` depois de gravar a coluna no schema. A função tenta de novo uma vez e o stream segue com o schema novo (decisão B4). Qualquer outro erro é relançado.
- **Retorno:** quantas linhas esta execução (`_run_id`) gravou na tabela.

Teste desta parte: só `category_translation` (71 linhas). As demais entidades rodam na Parte 4.

In [0]:
# O QUE FAZ: define quantas vezes o stream pode rodar e declara as funções que reconhecem o
#   erro de mudança de schema e se a entidade já tem arquivos na landing.
#
# FUNÇÕES DECLARADAS:
# - is_schema_change(error): True se o erro é o UnknownFieldException do Auto Loader. Pura: só
#   olha o texto do erro. A mensagem traz a classe UNKNOWN_FIELD_EXCEPTION (com subclasses como
#   NEW_FIELDS_IN_FILE) e, em algumas versões, o nome UnknownFieldException; por isso testa os dois.
# - has_source_files(entity): True se a landing já tem pelo menos um dia da entidade (pasta
#   sim_date=...). Sem nenhum arquivo, o Auto Loader não tem como inferir o schema e falha com
#   CF_EMPTY_DIR_FOR_SCHEMA_INFERENCE; é o caso de order_reviews nos primeiros dias, porque a
#   avaliação só é emitida quando o cliente responde, dias depois da entrega. A entidade é
#   ignorada até o primeiro arquivo chegar; nada se perde, porque o stream ainda não começou e,
#   quando começar, lê todos os arquivos existentes (includeExistingFiles = true, o padrão).
#
# RECURSOS:
# - str(error): texto da exceção. No PySpark, o erro do stream chega como
#   StreamingQueryException, com a causa original (o erro do Auto Loader) dentro da mensagem.
# - any(... for ... in ...): True se pelo menos um item da sequência for verdadeiro.
# - dbutils.fs.ls(pasta): lista a pasta; lança erro se ela não existe (a pasta da entidade só é
#   criada pelo simulador quando ele publica o primeiro arquivo), por isso o try/except.
MAX_ATTEMPTS = 2  # 1ª execução + 1 nova tentativa após mudança de schema

SCHEMA_CHANGE_MARKERS = ("UNKNOWN_FIELD_EXCEPTION", "UnknownFieldException")


def is_schema_change(error: Exception) -> bool:
    """True se o stream parou porque o Auto Loader encontrou coluna nova (addNewColumns)."""
    message = str(error)
    return any(marker in message for marker in SCHEMA_CHANGE_MARKERS)


def has_source_files(entity: str) -> bool:
    """True se a landing já tem pelo menos um dia da entidade."""
    try:
        return any(f.name.startswith("sim_date=") for f in dbutils.fs.ls(source_path(entity)))
    except Exception:
        return False  # a pasta da entidade ainda não existe

In [0]:
# O QUE FAZ: declara a execução de um stream da origem até a tabela bronze e a função de
#   ingestão com a nova tentativa em mudança de schema.
#
# FUNÇÕES DECLARADAS:
# - run_stream(entity, run_id): monta leitura + auditoria + escrita em Delta, inicia o stream
#   com availableNow e espera ele terminar. Se o stream falha, o erro aparece no
#   awaitTermination().
# - rows_written_by_run(entity, run_id): conta, na tabela, as linhas gravadas com este run_id.
#   Ler da tabela (e não das métricas do stream) mede o que de fato foi gravado.
# - ingest_entity(entity, run_id): chama run_stream até MAX_ATTEMPTS vezes. Repete só quando
#   o erro é mudança de schema; qualquer outro erro, ou a última tentativa, relança o erro.
#   Devolve as linhas gravadas por esta execução.
#
# FUNÇÕES NATIVAS:
# - df.writeStream: escritor de streaming (o equivalente incremental de df.write).
# - .format("delta"): grava em tabela Delta.
# - .outputMode("append"): só acrescenta linhas novas (modo da bronze, append-only).
# - .option("checkpointLocation", caminho): onde o stream guarda o progresso e a lista de
#   arquivos já lidos. É o que impede a duplicidade técnica.
# - .option("mergeSchema", "true"): permite que uma coluna nova (vinda do addNewColumns) seja
#   acrescentada à tabela Delta; sem isso, a gravação falharia por schema diferente da tabela.
# - .queryName(nome): nome do stream, exibido na interface e nas métricas.
# - .trigger(availableNow=True): processa os arquivos disponíveis em quantos micro-batches
#   forem necessários e encerra.
# - .toTable(nome): inicia o stream gravando na tabela do Unity Catalog (cria se não existir).
#   Devolve um StreamingQuery, o "controle" do stream em execução.
# - query.awaitTermination(): bloqueia até o stream terminar; relança o erro se ele falhou.
# - spark.table(nome): lê a tabela como DataFrame (batch).
# - range(1, n + 1): 1, 2, ..., n. continue: vai para a próxima volta do for.
# - raise (sozinho): relança o mesmo erro, com o traceback original.
def run_stream(entity: str, run_id: str) -> None:
    """Executa o stream da entidade até processar todos os arquivos disponíveis."""
    query = (
        add_audit_columns(read_entity_stream(entity), run_id)
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path(entity))
        .option("mergeSchema", "true")
        .queryName(f"bronze_{entity}")
        .trigger(availableNow=True)
        .toTable(table_name(entity))
    )
    query.awaitTermination()


def rows_written_by_run(entity: str, run_id: str) -> int:
    """Linhas da tabela bronze da entidade gravadas pela execução run_id."""
    return spark.table(table_name(entity)).where(F.col("_run_id") == run_id).count()


def ingest_entity(entity: str, run_id: str) -> int:
    """Ingere os arquivos novos da entidade; nova tentativa só em mudança de schema."""
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            run_stream(entity, run_id)
            return rows_written_by_run(entity, run_id)
        except Exception as error:
            if attempt < MAX_ATTEMPTS and is_schema_change(error):
                print(f"{entity}: schema mudou (coluna nova); nova tentativa {attempt + 1}/{MAX_ATTEMPTS}")
                continue
            raise

In [0]:
# O QUE FAZ: roda a ingestão de uma entidade pequena (71 linhas, só no dia 2026-09-06) para
#   validar a função antes de rodar as 10 entidades na Parte 4.
#   Efeito: cria a tabela olist.bronze.category_translation e o checkpoint da entidade.
#   Na Parte 4, esta entidade roda de novo e deve gravar 0 linhas (o arquivo já foi lido).
linhas = ingest_entity("category_translation", RUN_ID)
print(f"category_translation: {linhas} linhas gravadas (esperado: 71)")

In [0]:
# O QUE FAZ: mostra uma amostra da tabela criada e um resumo das colunas de auditoria:
#   todas as linhas devem ter o mesmo arquivo de origem, sim_date 2026-09-06, este run_id e
#   _rescued_data vazio.
#
# FUNÇÕES NATIVAS:
# - df.groupBy(...).agg(...): agrupa e calcula agregações por grupo.
# - F.count("*"): número de linhas do grupo.
# - F.count_if(condição): linhas em que a condição é verdadeira.
# - F.min / F.max: menor e maior valor (aqui, horários de ingestão).
bronze = spark.table(table_name("category_translation"))

display(bronze.limit(5))

display(
    bronze
    .groupBy("sim_date", "_source_file", "_run_id")
    .agg(
        F.count("*").alias("linhas"),
        F.count_if(F.col("_rescued_data").isNotNull()).alias("com_rescued_data"),
        F.min("_ingested_at").alias("primeira_ingestao"),
        F.max("_source_file_modified_at").alias("arquivo_modificado_em"),
    )
)

## Execução, documentação e validação

- **Execução:** as 10 entidades, uma por vez. Entidade sem nenhum arquivo na landing ainda (ex.: `order_reviews` nos primeiros dias) é ignorada até o primeiro arquivo chegar. Uma entidade com erro não impede as outras de serem capturadas; ao final, se alguma falhou, a célula termina com erro listando quais.
- **Documentação:** comentário em cada tabela e nas colunas técnicas (`sim_date`, `_rescued_data`, `_corrupt_record` e auditoria). Só é gravado o que mudou, para não criar uma versão nova da tabela a cada execução. As colunas de negócio são documentadas na silver, junto com o contrato de tipos.
- **Validação (aceite da 1.3):** contagens da bronze por entidade e dia iguais às linhas dos arquivos em `raw_data`; uma nova execução sem arquivos novos grava 0 linhas.
- **Linhas inválidas:** lista os registros com `_corrupt_record` preenchida, para conferir com o gabarito (`corrupt_file`).

In [0]:
# O QUE FAZ: define os textos de documentação e declara a função que os aplica à tabela,
#   gravando só os comentários que estão diferentes (idempotente e sem versões desnecessárias).
#
# FUNÇÕES DECLARADAS:
# - sql_string(text): transforma um texto Python em literal SQL entre aspas simples, escapando
#   barra invertida e aspas. Necessário porque COMMENT não aceita parameter marker; os textos
#   são constantes deste notebook, não entrada de usuário.
# - document_table(entity): compara o comentário atual da tabela e de cada coluna técnica com o
#   texto esperado e executa só os comandos necessários. Devolve quantos comandos executou
#   (0 nas execuções seguintes à primeira).
#
# FUNÇÕES NATIVAS E SQL:
# - spark.catalog.getTable(nome).description: comentário atual da tabela (None se não houver).
# - spark.table(nome).schema.fields: lista de colunas (StructField) da tabela; cada uma tem
#   name, dataType e metadata. O comentário da coluna fica em metadata["comment"].
# - dict.get(chave): valor da chave, ou None se não existir (sem erro).
# - COMMENT ON TABLE t IS '...': grava o comentário da tabela.
# - ALTER TABLE t ALTER COLUMN `c` COMMENT '...': grava o comentário da coluna. As crases
#   protegem nomes que começam com "_".
# - str.replace(a, b): troca todas as ocorrências de a por b.
BRONZE_TABLE_COMMENTS = {
    "orders": "Cabeçalho dos pedidos: dados conhecidos no momento da compra.",
    "order_status_events": "Eventos de mudança de status dos pedidos (created, approved, shipped, delivered e status finais).",
    "order_items": "Itens dos pedidos: produto, vendedor, preço e frete.",
    "order_payments": "Pagamentos dos pedidos: forma, parcelas e valor.",
    "order_reviews": "Avaliações dos pedidos feitas pelos clientes.",
    "customers": "Feed de mudanças cadastrais de clientes (CDC).",
    "sellers": "Feed de mudanças cadastrais de vendedores (CDC).",
    "products": "Cadastro de produtos (carga inicial).",
    "geolocation": "Prefixos de CEP com coordenadas, cidade e estado (carga inicial).",
    "category_translation": "Tradução das categorias de produto para o inglês (carga inicial).",
}

BRONZE_TABLE_SUFFIX = " Bronze: dado como recebido da landing (raw_data), append-only, colunas de negócio como string."

TECHNICAL_COLUMN_COMMENTS = {
    "sim_date": "Dia simulado do arquivo de origem (AAAA-MM-DD), lido do nome da pasta sim_date=.",
    "_rescued_data": "Valores que não se encaixaram no schema, em JSON; nulo quando o registro foi lido por inteiro.",
    "_corrupt_record": "Linha original do arquivo quando não é um JSON válido (demais colunas nulas); nulo quando o registro foi lido normalmente.",
    "_ingested_at": "Momento em que o registro foi gravado na bronze.",
    "_source_file": "Caminho do arquivo de origem na landing.",
    "_source_file_modified_at": "Momento em que o arquivo de origem foi gravado na landing; com _ingested_at, mede o atraso de ingestão.",
    "_run_id": "Execução da bronze que gravou o registro (job.run_id no job; UUID na execução interativa).",
}


def sql_string(text: str) -> str:
    """Literal SQL entre aspas simples, com barra invertida e aspas escapadas."""
    return "'" + text.replace("\\", "\\\\").replace("'", "\\'") + "'"


def document_table(entity: str) -> int:
    """Aplica os comentários da tabela e das colunas técnicas que estiverem diferentes."""
    name = table_name(entity)
    statements = []

    table_comment = BRONZE_TABLE_COMMENTS[entity] + BRONZE_TABLE_SUFFIX
    if spark.catalog.getTable(name).description != table_comment:
        statements.append(f"COMMENT ON TABLE {name} IS {sql_string(table_comment)}")

    for field in spark.table(name).schema.fields:
        comment = TECHNICAL_COLUMN_COMMENTS.get(field.name)
        if comment and field.metadata.get("comment") != comment:
            statements.append(f"ALTER TABLE {name} ALTER COLUMN `{field.name}` COMMENT {sql_string(comment)}")

    for statement in statements:
        spark.sql(statement)
    return len(statements)

In [0]:
# O QUE FAZ: ingere e documenta as 10 entidades, uma por vez, e imprime o resumo.
#   Se uma entidade falha, o erro é guardado e as próximas continuam; no fim, se houve falha, a
#   célula lança RuntimeError com a lista (no job, a task termina como falha).
#   Entidade sem arquivos na landing (has_source_files = False) é ignorada, sem erro.
#   category_translation já foi lida na Parte 3: mostra 71 se o run_id for o mesmo da Parte 3
#   (a contagem é por run_id) ou 0 se for um run_id novo. Nenhum dos dois casos duplica dados.
#
# RECURSOS:
# - dicionários resultados e falhas: entidade -> linhas gravadas / mensagem de erro.
# - try/except dentro do for: trata o erro de uma entidade sem interromper o loop.
# - f"{n:>10,}": alinha à direita em 10 posições com separador de milhar.
# - sum(dict.values()): soma das linhas de todas as entidades.
resultados, falhas = {}, {}

for entity in ENTITIES:
    if not has_source_files(entity):
        print(f"{entity:22} sem arquivos na landing ainda: ignorada nesta execução")
        continue
    try:
        resultados[entity] = ingest_entity(entity, RUN_ID)
        comandos = document_table(entity)
        print(f"{entity:22} {resultados[entity]:>10,} linhas  ({comandos} comentários aplicados)")
    except Exception as error:
        falhas[entity] = f"{type(error).__name__}: {error}"[:500]
        print(f"{entity:22} FALHOU: {falhas[entity]}")

print(f"\nTotal desta execução: {sum(resultados.values()):,} linhas")

if falhas:
    raise RuntimeError(f"Falha na ingestão de {len(falhas)} entidade(s): {', '.join(falhas)}")

In [0]:
# O QUE FAZ: compara, por entidade e dia, as linhas da bronze com as linhas dos arquivos de
#   raw_data (JSON Lines: 1 linha = 1 registro). Critério de aceite da 1.3: tudo "ok" e total
#   de 1.036.291 linhas (1.036.286 de 2026-09-06 + 5 de 2026-09-07).
#   Só entram as entidades que já têm arquivos na landing (as outras não têm o que comparar).
#
# FUNÇÃO DECLARADA:
# - counts_by_day(df, entity): linhas por sim_date, com a entidade como coluna. sim_date vira
#   string nos dois lados (na leitura de texto, a partição é inferida como date).
#
# FUNÇÕES NATIVAS:
# - spark.read.text(pasta): lê os arquivos como texto; descobre a partição sim_date pelas pastas.
# - functools.reduce(DataFrame.unionByName, lista): empilha os DataFrames da lista um a um.
# - .cast("string"): converte o tipo da coluna.
# - df.join(outro, on=[...], how="full"): full outer join; mantém as linhas dos dois lados,
#   assim um dia que existe só na landing (ou só na bronze) aparece com o outro lado nulo.
# - F.coalesce(a, b): primeiro valor não nulo (dia ausente em um lado conta como 0).
# - F.sum(...): soma dos valores; F.count_if: linhas em que a condição é verdadeira.
from functools import reduce


def counts_by_day(df: DataFrame, entity: str) -> DataFrame:
    return df.groupBy(F.col("sim_date").cast("string").alias("sim_date")).count().withColumn("entidade", F.lit(entity))


com_arquivos = [e for e in ENTITIES if has_source_files(e)]

landing_counts = reduce(DataFrame.unionByName, [counts_by_day(spark.read.text(source_path(e)), e) for e in com_arquivos])
bronze_counts = reduce(DataFrame.unionByName, [counts_by_day(spark.table(table_name(e)), e) for e in com_arquivos])

comparacao = (
    landing_counts.withColumnRenamed("count", "landing")
    .join(bronze_counts.withColumnRenamed("count", "bronze"), on=["entidade", "sim_date"], how="full")
    .withColumn("ok", F.coalesce(F.col("landing"), F.lit(0)) == F.coalesce(F.col("bronze"), F.lit(0)))
    .orderBy("entidade", "sim_date")
)

display(comparacao)
display(comparacao.agg(
    F.sum("landing").alias("total_landing"),
    F.sum("bronze").alias("total_bronze"),
    F.count_if(~F.col("ok")).alias("divergencias"),
))

In [0]:
# O QUE FAZ: lista os registros cuja linha de origem não era um JSON válido (_corrupt_record
#   preenchida), em todas as entidades com tabela. Compare com o gabarito: cada linha aqui deve
#   corresponder a uma falha corrupt_file em ops.simulator_chaos_log.
#
# FUNÇÕES NATIVAS:
# - spark.catalog.tableExists(nome): True se a tabela existe (entidade já ingerida).
# - "coluna" in df.columns: protege contra uma tabela ainda sem a coluna.
# - F.lit(valor).alias(nome): coluna constante com o nome da entidade.
# - reduce(DataFrame.unionByName, lista): empilha os resultados de todas as entidades.
corrompidos = reduce(DataFrame.unionByName, [
    spark.table(table_name(e))
    .where(F.col("_corrupt_record").isNotNull())
    .select(F.lit(e).alias("entidade"), "sim_date", "_source_file", "_corrupt_record")
    for e in ENTITIES
    if spark.catalog.tableExists(table_name(e)) and "_corrupt_record" in spark.table(table_name(e)).columns
])

display(corrompidos)


In [0]:
# O QUE FAZ: roda de novo, com outro run_id, as entidades que têm arquivos. Como nenhum arquivo novo chegou,
#   o checkpoint faz cada stream terminar sem ler nada: todas devem gravar 0 linhas. É a prova
#   de que reexecutar a bronze não gera duplicidade técnica.
#
# RECURSOS:
# - uuid.uuid4(): run_id novo, só para este teste (as linhas gravadas seriam desse run_id).
# - {e: ... for e in ENTITIES}: dict comprehension com o resultado de cada entidade.
# - all(...): True se todos os itens forem verdadeiros.
run_id_teste = str(uuid.uuid4())
reexecucao = {entity: ingest_entity(entity, run_id_teste) for entity in ENTITIES if has_source_files(entity)}

for entity, linhas in reexecucao.items():
    print(f"{entity:22} {linhas} linhas")
print("\nReexecução sem duplicidade:", "OK" if all(n == 0 for n in reexecucao.values()) else "FALHOU")

In [0]:
# O QUE FAZ: mostra as colunas de olist.bronze.orders com tipo e comentário, e o comentário da
#   tabela, para conferir a documentação aplicada.
# - DESCRIBE TABLE t: colunas, tipos e comentários.
display(spark.sql(f"DESCRIBE TABLE {table_name('orders')}"))
print(spark.catalog.getTable(table_name("orders")).description)